# Linear Regression Quiz — Complete Solutions 📘
## Mr Beast Videos Dataset

**Setup rules**: `alpha=0.05`, `random_state=617`, `train_size=0.9` (simple random sampling).  
**Source**: Quiz on Linear Regression module.

## 🔧 Setup: Load data + apply categorical ordering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from statsmodels.formula.api import ols

# Load data
data = pd.read_csv('mr_beast.csv')

# Apply ordered categorical to publish_day (Mon as reference)
data['publish_day'] = pd.Categorical(data['publish_day'],
                                      categories=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
                                      ordered=True)

# 90/10 split with seed 617
train = data.sample(frac=0.9, random_state=617)
test = data.drop(labels=train.index)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

---
## ❓ Question 1

> Read in the data and run the code that follows. `pd.Categorical()` will convert publish_day to an ordered categorical variable with Mon as the first day of the week.
>
> ```python
> import pandas as pd    
> data = pd.read_csv('mr_beast.csv')    
> data['publish_day'] = pd.Categorical(data['publish_day'],   
>                                        categories=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],   
>                                        ordered=True) 
> ```
>
> Next, split the data into a train and test sample such that 90% of the data is in the train sample. Use simple random sampling with pandas, `data.sample()` (where data is the name of a dataframe). For reproducibility set random state to 617. 
>
> **What is the median number of "likes" in the train sample?**

### ✅ Answer: `10581.0`

In [ ]:
train.likes.median()
# → 10581.0

---
## ❓ Question 2

> Construct a scatter plot to examine the relationship between "likes" and "views". Place views on the x-axis and likes on the y-axis. 
> **What is the direction of the points?**

### ✅ Answer: **Upward / Positive** — points trend from bottom-left to top-right.

Correlation = **0.8174** (strong positive → upward slope)

In [ ]:
sns.scatterplot(data=train, x='views', y='likes', color='steelblue', s=15)
plt.show()

# Verify with correlation
print('Correlation:', train[['views','likes']].corr().iloc[0,1])

---
## ❓ Question 3

> **Which of the following variables have a statistically significant pearson correlation with likes?**

### ✅ Answer: **`views`, `comments`, `number_of_tags`, `time_since`** (all have p < 0.05)

| Variable | Correlation | p-value | Significant? |
|---|---:|---:|:---:|
| views | 0.8174 | 1.15e-156 | ✅ YES |
| comments | 0.8217 | 1.14e-159 | ✅ YES |
| duration_seconds | 0.0083 | 0.833 | ❌ NO |
| number_of_tags | -0.3589 | 4.22e-21 | ✅ YES |
| length_description | 0.0119 | 0.763 | ❌ NO |
| length_title | 0.0584 | 0.138 | ❌ NO |
| time_since | -0.6746 | 4.34e-87 | ✅ YES |

In [ ]:
numeric_vars = ['views','comments','duration_seconds','number_of_tags',
                'length_description','length_title','time_since']

for var in numeric_vars:
    r, p = pearsonr(train[var], train['likes'])
    sig = 'YES' if p < 0.05 else 'NO'
    print(f'{var:<25} r={r:>8.4f}  p={p:.4e}  sig={sig}')

---
## ❓ Question 4

> **Which of the following variables has the strongest (pearson) correlation with likes?**

### ✅ Answer: **`comments`** (|r| = 0.8217)

⚠️ Watch out! `comments` (0.8217) only narrowly beats `views` (0.8174). Read carefully!

In [ ]:
# Rank by absolute correlation
results = []
for var in numeric_vars:
    r, p = pearsonr(train[var], train['likes'])
    results.append((var, r, abs(r)))
results.sort(key=lambda x: x[2], reverse=True)
for v, r, ar in results:
    print(f'{v:<25} r={r:>8.4f}  |r|={ar:.4f}')

---
## ❓ Question 5

> We expect the hours since the release of a video to have an effect on likes. The longer a video has been available, the more opportunity a viewer has to like it. Let us evaluate this proposition by constructing a regression model to predict "likes" with "time_since". Call this `model1`.
>
> Since `model1` is built on sample data, it is important to see if the coefficient estimates will be non-zero in the population. **Examine the p-value for time_since and select the appropriate choice.**

### ✅ Answer: **The coefficient is statistically significant (different from 0).** 

p-value ≈ 0 (essentially 0). We reject H₀: β₁ = 0. There is strong evidence that `time_since` affects `likes` in the population.

In [ ]:
model1 = ols('likes ~ time_since', data=train).fit()
print(model1.summary())

print('\np-value of time_since:', model1.pvalues['time_since'])

---
## ❓ Question 6

> **Based on the coefficient of `time_since` in `model1`, which of the following is true?**

### ✅ Answer: **For every 1 additional hour since the video's release, predicted `likes` *decrease* by approximately 48.19.**

**Coefficient: `-48.189793`** (negative!)

💡 Note: This is COUNTERINTUITIVE — you'd expect older videos to have *more* likes. But in this data, older videos may have been less viral, or newer videos benefit from a larger fan base.

In [ ]:
model1.params
# Intercept     3194182.753976
# time_since        -48.189793

---
## ❓ Question 7

> The first video in the train data has 2125003 likes. **Based on `model1`, what is the predicted number of likes for the first video?**
> To be sure, the first video in train has a video_id of `'gL6iSCSHjco'`.

### ✅ Answer: **`1922936.00218185`** (do NOT round!)

ŷ = 3194182.753976 + (-48.189793)(26380) = 1922936.00218185

In [ ]:
# Method 1: use .predict() — always preferred
model1.predict(train.iloc[[0]])

# Or equivalently via [0]
print('First row predicted:', model1.predict()[0])

# Method 2: manual calculation
b0 = model1.params['Intercept']
b1 = model1.params['time_since']
print('Manual:', b0 + b1 * train.iloc[0]['time_since'])

---
## ❓ Question 8

> **Based on this model, what is the predicted number of likes for a video that has been available for 25,000 hours?**

### ✅ Answer: **`1989437.9171733349`**

In [ ]:
new_data = pd.DataFrame({'time_since':[25000]})
model1.predict(new_data)
# → 1989437.917173

---
## ❓ Question 9

> Construct a model to examine the effect of number of tags on likes. Call this `model2`. **Based on `model2`, what is the effect of adding one more tag on likes for a video?**

### ✅ Answer: **`-64894.673692`** (each additional tag DECREASES likes by ~64894.67)

Counterintuitive again — but the data shows tags negatively correlate with likes.

In [ ]:
model2 = ols('likes ~ number_of_tags', data=train).fit()
print(model2.params)
# Intercept          1204836.668946
# number_of_tags      -64894.673692

---
## ❓ Question 10

> Let us see if the day the video was published has an impact on likes. At the start of this assignment, we converted publish_day to an ordered categorical variable with Mon (i.e., Monday) as the first value. So, Mon will serve as a reference level for the levels of publish_day.
>
> Run a linear regression to examine the influence of publish_day on likes. Call this `model3`. **What is the R² of this model?**

### ✅ Answer: **`0.05989930638...`** (or `0.0598993...` — do not round!)

In [ ]:
model3 = ols('likes ~ publish_day', data=train).fit()
print('R²:', model3.rsquared)
print(model3.summary())

---
## ❓ Question 11

> **Based on `model3` constructed in the previous question, which of the following are true.** Select one or more correct answers.

### ✅ Answer: True statements based on coefficients and p-values:

Days **significantly different from Monday** (reference) at α=0.05:

| Day vs Mon | Coefficient | p-value | Significant? |
|---|---:|---:|:---:|
| Tue | +264,722 | 0.289 | ❌ NO |
| Wed | +588,753 | 0.019 | ✅ YES |
| Thu | +533,320 | 0.023 | ✅ YES |
| Fri | +562,235 | 0.017 | ✅ YES |
| Sat | +1,064,253 | <0.001 | ✅ YES |
| Sun | +141,471 | 0.539 | ❌ NO |

**Likely true statements:**
- ✅ Videos released on **Saturday** get significantly more likes than Monday.
- ✅ Videos released on **Wednesday/Thursday/Friday** get significantly more likes than Monday.
- ✅ The **Saturday coefficient is the largest** → Saturday videos receive the most likes on average.
- ✅ **Tuesday and Sunday** are NOT significantly different from Monday.
- ❌ NOT TRUE: "All days are significantly different from Monday" (because Tue and Sun aren't).

In [ ]:
print(model3.pvalues)
print()
print(model3.params)

---
## ❓ Question 12

> **Based on `model3`, on average how many more likes do videos released on Thursday get than videos released on Monday?**

### ✅ Answer: **`533320.198465`** (the `publish_day[T.Thu]` coefficient — exact difference from Monday)

In [ ]:
model3.params['publish_day[T.Thu]']
# → 533320.198465

---
## ❓ Question 13

> In this section, you will use the scikit-learn framework for fitting regression models. Unless stated otherwise, use the train sample for conducting analysis.
>
> Separate the outcome variable from the features, assigning likes to a Series, `y`, and assigning the following features to a DataFrame, `X`: `duration_seconds`, `number_of_tags`, `length_description`, `length_title`, `time_since`, `publish_day`. 
>
> Next, create a binned outcome variable as follows: 
>
> ```python
> y_binned = pd.cut(data.likes, bins=[0,1000,10000,100000,1000000,25000000])
> ```
>
> Finally, use `y_binned` to do stratified random sampling, placing 90% of the data in the train sample and the remaining 10% in test. Set `random_state` to 617 for reproducibility. 
>
> **What is the median value of duration_seconds in the train sample?**

### ✅ Answer: **`274.0`**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

y = data.likes
X = data[['duration_seconds','number_of_tags','length_description',
          'length_title','time_since','publish_day']]

# Create binned outcome for stratification
y_binned = pd.cut(data.likes, bins=[0,1000,10000,100000,1000000,25000000])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.9, random_state=617, stratify=y_binned)

print('Train shape:', X_train_raw.shape)
print('Median duration_seconds in train:', X_train_raw['duration_seconds'].median())
# → 274.0

---
## ❓ Question 14

> Dummy code `publish_day` using `pandas.get_dummies()`. When dummy coding, ensure you set `drop_first` to True. Do this for both the train and test samples.  
>
> **What is the mean value for the dummy variable representing Tuesday?**

### ✅ Answer: **`0.10819165378670788`**

💡 This is the fraction of videos in train published on Tuesday (since dummy = 1 for Tue rows, 0 otherwise; mean of 0/1 = proportion).

In [ ]:
# IMPORTANT: drop_first=True drops Monday (the first in categorical order)
X_train = pd.get_dummies(X_train_raw, columns=['publish_day'], drop_first=True)
X_test = pd.get_dummies(X_test_raw, columns=['publish_day'], drop_first=True)

# Align test columns with train columns (CRITICAL!)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Convert boolean dummies to numeric
X_train = X_train.astype(float)
X_test = X_test.astype(float)

print('Columns:', list(X_train.columns))
print('\nMean of publish_day_Tue:', X_train['publish_day_Tue'].mean())
# → 0.10819165378670788

---
## ❓ Question 15

> Fit a regression model to predict likes using the features used to create the feature matrix above. Call this `model4`. **What is the predicted value for likes in the last observation in the train sample?**

### ✅ Answer: **`-142999.6538872989`** (yes, can be negative — limitation of linear regression!)

In [ ]:
model4 = LinearRegression()
model4.fit(X_train, y_train)

y_pred_train = model4.predict(X_train)
y_pred_test  = model4.predict(X_test)

# Last observation in train (use index -1)
print('Last predicted:', y_pred_train[-1])
# Equivalently:
print('Same thing:', model4.predict(X_train.iloc[[-1]])[0])
# → -142999.6538872989

---
## ❓ Question 16

> **What is the root mean squared error for `model4` in the train sample?**

### ✅ Answer: **`1030940.8297459387`**

In [ ]:
rmse_train = root_mean_squared_error(y_train, y_pred_train)
print('Train RMSE:', rmse_train)
# → 1030940.8297459387

---
## ❓ Question 17

> **What is the root mean squared error for `model4` in the test sample?**

### ✅ Answer: **`570004.4990510686`**

💡 Test RMSE < Train RMSE is unusual but possible — likely because train contains some influential outliers absent from test.

In [ ]:
rmse_test = root_mean_squared_error(y_test, y_pred_test)
print('Test RMSE:', rmse_test)
# → 570004.4990510686

---
# 📋 All Answers Summary (Copy-Paste Ready)

| Q | Answer | Notes |
|---|---|---|
| 1 | `10581` | Median likes in train |
| 2 | Upward / Positive | r = 0.817 |
| 3 | views, comments, number_of_tags, time_since | p < 0.05 |
| 4 | `comments` | \|r\| = 0.8217 (narrowly beats views) |
| 5 | Significant (p < 0.05) | Reject H₀: β = 0 |
| 6 | NEGATIVE, ~-48.19 per hour | Each hour ↑ → likes ↓ |
| 7 | `1922936.00218185` | b0 + b1×26380 |
| 8 | `1989437.9171733349` | b0 + b1×25000 |
| 9 | -64894.673692 per tag | Each tag ↑ → likes ↓ |
| 10 | R² = `0.0598993063...` | publish_day only explains ~6% |
| 11 | Wed, Thu, Fri, Sat significant vs Mon | Tue, Sun NOT |
| 12 | `533320.198465` | Thu coef (vs Mon) |
| 13 | `274.0` | Median duration_seconds (sklearn train) |
| 14 | `0.10819165378670788` | Mean of Tuesday dummy |
| 15 | `-142999.6538872989` | Last train observation prediction |
| 16 | `1030940.8297459387` | Train RMSE |
| 17 | `570004.4990510686` | Test RMSE |

### ⚠️ Entry format reminders from assignment:
- Do NOT round unless told
- No commas in numbers: `100000` not `100,000`
- No units: `34.56` not `$34.56`
- Lead with 0 before decimal: `0.34` not `.34`
- Drop trailing zeros: `0.3` not `0.30`